In [1]:
# load usda_data_clean2.csv 
import pandas as pd 
df_usda = pd.read_csv('usda_data_clean2.csv')
df_usda.head()

FileNotFoundError: [Errno 2] No such file or directory: 'usda_data_clean2.csv'

In [ ]:
# load off_data_clean2.csv
df_off = pd.read_csv('off_data_clean2.csv')
df_off.head()

In [ ]:
#find out how many unquie values are in category column in df
print(df_usda['cat'].nunique())
df_off['cat'].nunique()

In [ ]:
#check how many entries are in each category in df_usda
print(df_usda['cat'].value_counts())

In [ ]:
#how many cat only have 1 entry in df_usda
print((df_usda['cat'].value_counts() == 1).sum())

In [ ]:
#how many entries are in each category in df_off
print(df_off['cat'].value_counts())
#how many cat only have 1 entry in df_off
print((df_off['cat'].value_counts() == 1).sum())

In [ ]:
#Normalize: lowercase, strip hyphens → spaces
df_off['cat'] = df_off['cat'].str.lower().str.replace('-', ' ')

In [ ]:
#normalize usda dataset the same way
df_usda['cat'] = df_usda['cat'].str.lower().str.replace('-', ' ')

In [ ]:
#how many words are in each category in df_usda
df_usda['cat_word_count'] = df_usda['cat'].str.split().apply(len)
print(df_usda['cat_word_count'].value_counts())

In [ ]:
#drop all categories with more than 5 words in df_usda
df_usda = df_usda[df_usda['cat_word_count'] <= 6]

In [ ]:
#show category count in df_usda
print(df_usda['cat_word_count'].value_counts())


In [ ]:
#make all entries in "item_name" column a string
df_usda['item_name'] = df_usda['item_name'].astype(str)

In [ ]:
#show float types in "item_name" column in df_usda
print(df_usda[df_usda['item_name'].apply(lambda x: isinstance(x, float))]['item_name'])

In [ ]:
#remove all entries with float types in "item_name" column in df_usda
df_usda = df_usda[~df_usda['item_name'].apply(lambda x: isinstance(x, float))]

In [ ]:
#drop all entries with more than 5 words in item_name column in df_usda
df_usda['item_name_word_count'] = df_usda['item_name'].str.split().apply(len)
df_usda = df_usda[df_usda['item_name_word_count'] <= 5]

In [ ]:
#show categoriy with 110 words in df_usda
print(df_usda[df_usda['cat_word_count'] == 110]['cat'].values)

In [ ]:
#how many entries are now in the df_usda dataset
print(len(df_usda))

In [ ]:
#identify duplicates or near duplicates in df_usda
duplicates = df_usda[df_usda.duplicated(subset=['item_name'], keep=False)]
duplicates['item_name'].value_counts()

#identify near duplicates in df_usda using fuzzy matching
from fuzzywuzzy import fuzz
from fuzzywuzzy import process  

# Create a list of unique item names
item_names = df_usda['item_name'].unique()  

In [ ]:
#clean off dataset the same way as usda dataset
df_off['item_name'] = df_off['item_name'].astype(str)
df_off['item_name_word_count'] = df_off['item_name'].str.split().apply(len)
df_off = df_off[df_off['item_name_word_count'] <= 5]


In [ ]:
#remove "food_measurements" from usda dataset
food_measurements = [
    # Weight / mass
    "mg", "milligram", "milligrams",
    "g", "gram", "grams",
    "kg", "kilogram", "kilograms",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",

    # Volume
    "ml", "milliliter", "milliliters", "millilitre", "millilitres",
    "cl", "centiliter", "centiliters", "centilitre", "centilitres",
    "l", "liter", "liters", "litre", "litres",
    "tsp", "teaspoon", "teaspoons",
    "tbsp", "tablespoon", "tablespoons",
    "fl oz", "fluid ounce", "fluid ounces",
    "cup", "cups",
    "pint", "pints",
    "quart", "quarts",
    "gallon", "gallons",
    "dash", "dashes",
    "pinch", "pinches",
    "splash", "splashes",
    "drop", "drops",

    # Count / piece-based
    "piece", "pieces",
    "pc", "pcs",
    "unit", "units",
    "item", "items",
    "whole", "halves", "half",
    "quarter", "quarters",
    "slice", "slices",
    "stick", "sticks",
    "cube", "cubes",
    "chunk", "chunks",
    "wedge", "wedges",
    "strip", "strips",
    "ring", "rings",
    "clove", "cloves",
    "leaf", "leaves",
    "sprig", "sprigs",
    "stalk", "stalks",
    "stem", "stems",
    "head", "heads",
    "bunch", "bunches",
    "bulb", "bulbs",
    "ear", "ears",
    "kernel", "kernels",
    "pod", "pods",
    "bean", "beans",
    "egg", "eggs",
    "fillet", "fillets",
    "breast", "breasts",
    "thigh", "thighs",
    "drumstick", "drumsticks",
    "leg", "legs",
    "wing", "wings",

    # Prepared-food / serving style
    "serving", "servings",
    "portion", "portions",
    "helping", "helpings",
    "plate", "plates",
    "bowl", "bowls",
    "dish", "dishes",
    "tray", "trays",

    # Packaging / container-based
    "pack", "packs",
    "packet", "packets",
    "package", "packages",
    "bag", "bags",
    "box", "boxes",
    "carton", "cartons",
    "can", "cans",
    "tin", "tins",
    "jar", "jars",
    "bottle", "bottles",
    "tube", "tubes",
    "sachet", "sachets",
    "wrapper", "wrappers",
    "container", "containers",
    "cupful", "cupfuls",

    # Bakery / produce / retail style
    "loaf", "loaves",
    "roll", "rolls",
    "bun", "buns",
    "patty", "patties",
    "link", "links",
    "sausage", "sausages",
    "ball", "balls",
    "bar", "bars",
    "block", "blocks",

    # Descriptive prep amounts often used in recipes
    "handful", "handfuls",
    "fistful", "fistfuls",
    "scoop", "scoops",
    "ladle", "ladles",
    "spoonful", "spoonfuls",
    "heaped teaspoon", "heaped teaspoons",
    "heaped tablespoon", "heaped tablespoons",
    "level teaspoon", "level teaspoons",
    "level tablespoon", "level tablespoons",
    "to taste"
]

#remove all entries in "item_name" column in df_usda that contain any of the words in food_measurements
df_usda = df_usda[~df_usda['item_name'].str.contains('|'.join(food_measurements))]    

#remove number values from "item_name" column in df_usda
df_usda = df_usda[~df_usda['item_name'].str.contains(r'\d')]

#how many entries are now in the df_usda dataset
print(len(df_usda))



***Normalize Item Names***

In [ ]:
#normalize item names in df_usda
df_usda['item_name'] = df_usda['item_name'].str.lower().str.replace('-', ' ')

#replace mulitple spaces with a single space in "item_name" column in df_usda
df_usda['item_name'] = df_usda['item_name'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
#get rid of sonderzeichen in "item_name" column in df_usda
df_usda['item_name'] = df_usda['item_name'].str.replace(r'[^\w\s]', '', regex=True)

In [ ]:
#trim leading and trailing spaces in "item_name" column in df_usda
df_usda['item_name'] = df_usda['item_name'].str.strip()

**Chatty Ansatz**

In [ ]:
#normalize 
import re
import unicodedata
import pandas as pd

def normalize_item_name(s):
    if pd.isna(s):
        return ""

    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    tokens = s.split()
    tokens = sorted(set(tokens))   # important change

    return " ".join(tokens)

# USDA Deduplication Pipeline

## Goal
After the basic cleaning steps (lowercase, word-count filter, measurement removal, normalisation), the USDA dataset still contained a large number of near-duplicate `item_name` entries — e.g. `"apple"` vs `"apples"`, `"chicken breast"` vs `"chicken breasts"`. The deduplication pipeline collapses these into a single canonical name per cluster.

---

## Input
| Attribute | Value |
|---|---|
| Column used | `item_norm` (sort-normalised version of `item_name`) |

### What is `item_norm`?
Before deduplication, each `item_name` is transformed into a **sort-normalised** form:
```
unicode → ASCII  →  lowercase  →  remove special chars  →  tokenise  →  deduplicate tokens  →  sort tokens alphabetically  →  rejoin
```
Example: `"Chicken, Breast"` → `item_norm = "breast chicken"`

This ensures that `"chicken breast"` and `"breast chicken"` map to the same normalised form and are caught as exact duplicates before even reaching the fuzzy step.

---

## Deduplication Steps

### Step 1 — `item_norm` column
Each `item_name` is normalised into `item_norm` (sort-deduped tokens). An exact canonical map is built first: for each unique `item_norm`, the most frequent original `item_name` is selected as the representative.

### Step 2 — Build blocks (blocking)
To avoid O(n²) comparisons on 130k+ entries, names are grouped into **blocks** by a composite key:
```python
block_key = (first_4_chars_of_first_token, token_count, first_char_of_first_token)
```
Example: `"apple juice"` → block key `("appl", 2, "a")`

Only names within the same block are compared against each other.

### Step 3 — Fuzzy matching within blocks (RapidFuzz)
Within each block, every pair of names is compared using **`fuzz.token_set_ratio`**:
- **Threshold:** score ≥ 95
- `token_set_ratio` is order-insensitive and handles subset relationships
- All pairs above the threshold are recorded as *similar pairs*

### Step 4 — Build clusters (BFS graph traversal)
The similar pairs form an undirected graph. Connected components are found via **Breadth-First Search (BFS)**:
- Each node is a unique `item_norm`
- An edge exists if fuzzy score ≥ 95
- Each connected component = one **cluster** of near-duplicates

Example cluster:
```
- "apple"
- "apples"
- "apple fruit"
```

### Step 5 — Select a canonical name per cluster
For each cluster, one representative is chosen using this priority:
1. **Highest frequency** in the original dataset
2. **Shortest name** (tiebreaker — simpler is better)
3. **Alphabetical order** (final tiebreaker)

All other names in the cluster map to this canonical name. Names not in any cluster map to themselves.

### Step 6 — Apply mapping & deduplicate
- `item_norm` → `item_canonical` via `canonical_map_fuzzy`
- `drop_duplicates(subset=["item_canonical"])` removes remaining duplicate rows

### Step 7 — Remove duplicate words within names
A post-processing pass removes repeated tokens within a single name:
```
"chicken chicken breast"  →  "chicken breast"
```
Applied to both `item_name` and `item_canonical`.

### Step 8 — Finalise columns
- `item_name` is replaced with `item_canonical`
- Helper columns `item_norm` and `item_canonical` are dropped
- Saved to `usda_data_dedup_final.csv`

---

## Design Decisions
| Choice | Reason |
|---|---|
| Blocking before fuzzy | Avoids O(n²) comparisons on large datasets |
| `token_set_ratio` over `ratio` | Order-insensitive; handles partial overlaps robustly |
| Threshold = 95 | High precision — avoids merging different foods (e.g. `"pork"` ≠ `"pork rinds"`) |
| BFS clustering | Transitivity — if A≈B and B≈C, all three collapse into one cluster |
| Frequency-first canonical | Preserves the most commonly used real-world name |

In [ ]:
df_usda = df_usda.copy()
df_usda["item_norm"] = df_usda["item_name"].apply(normalize_item_name)

In [ ]:
print(df_usda[["item_name", "item_norm"]].head(20))
print("Rows:", len(df_usda))
print("Unique original:", df_usda["item_name"].nunique())
print("Unique normalized:", df_usda["item_norm"].nunique())

In [ ]:
canonical_map_exact = (
    df_usda.groupby("item_norm")["item_name"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)

df_usda["item_canonical"] = df_usda["item_norm"].map(canonical_map_exact)

Blocking Bauen


In [ ]:
df_unique = df_usda.drop_duplicates(subset=["item_norm"]).copy()

In [ ]:
def make_block_key(s):
    tokens = s.split()
    if not tokens:
        return ("", 0, "")

    first_token = tokens[0]
    token_count = len(tokens)
    prefix = first_token[:4]

    return (prefix, token_count, first_token[0])


from collections import defaultdict

blocks = defaultdict(list)

unique_norm_names = df_unique["item_norm"].dropna().unique()

for name in unique_norm_names:
    blocks[make_block_key(name)].append(name)


block_sizes = [len(v) for v in blocks.values()]
print("Anzahl Blöcke:", len(blocks))
print("Größter Block:", max(block_sizes))
print("Durchschnitt Blockgröße:", sum(block_sizes) / len(block_sizes))

Rapid Fuzzing auf blöcke

In [ ]:
pip install rapidfuzz

In [ ]:
from rapidfuzz import fuzz

threshold = 95
similar_pairs = []

for block_key, names in blocks.items():
    n = len(names)

    # winzige Blöcke kannst du direkt durchlaufen
    for i in range(n):
        for j in range(i + 1, n):
            score = fuzz.token_set_ratio(names[i], names[j])

            if score >= threshold:
                similar_pairs.append((names[i], names[j], score))

Aus Paren Cluster bauen

In [ ]:
from collections import defaultdict, deque

graph = defaultdict(set)

for a, b, score in similar_pairs:
    graph[a].add(b)
    graph[b].add(a)

visited = set()
clusters = []

for node in graph:
    if node not in visited:
        cluster = []
        queue = deque([node])
        visited.add(node)

        while queue:
            current = queue.popleft()
            cluster.append(current)

            for neighbor in graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        clusters.append(cluster)

Stufe 6 — pro Cluster einen Canonical Name wählen

In [ ]:
name_frequency = df_usda["item_norm"].value_counts().to_dict()

canonical_map_fuzzy = {}

for cluster in clusters:
    best_name = sorted(
        cluster,
        key=lambda x: (-name_frequency.get(x, 0), len(x), x)
    )[0]

    for name in cluster:
        canonical_map_fuzzy[name] = best_name

In [ ]:
for name in unique_norm_names:
    if name not in canonical_map_fuzzy:
        canonical_map_fuzzy[name] = name

Stufe 7 — Mapping auf den DataFrame anwenden

In [ ]:
df_usda["item_canonical"] = df_usda["item_norm"].map(canonical_map_fuzzy)

In [ ]:
df_usda_dedup = df_usda.drop_duplicates(subset=["item_canonical"]).copy()

Stufe 8 — Qualität prüfen, bevor du irgendwas final löschst

In [ ]:
print("Unique normalized:", df_usda["item_norm"].nunique())
print("Unique canonical:", df_usda["item_canonical"].nunique())

In [ ]:
#beispiel cluster

for i, cluster in enumerate(clusters[:20]):
    print(f"\nCluster {i+1}:")
    for name in cluster:
        print(" -", name)

In [ ]:
#show declutered dataframe
print(df_usda_dedup[["item_name", "item_norm", "item_canonical"]].head(20))

In [ ]:
#show entrie for "avocado" in df_usda_dedup with all cloumns
print(df_usda_dedup[df_usda_dedup['item_canonical'] == 'avocado'])

In [ ]:
#create csv from ds_usda_dedup
df_usda_dedup.to_csv('usda_data_dedup.csv', index=False)

In [ ]:
#litte eda on usda_data_dedup
print(df_usda_dedup['cat'].value_counts())

In [ ]:
#are there any duplicates in item_canonical column in df_usda_dedup
duplicates_canonical = df_usda_dedup[df_usda_dedup.duplicated(subset=['item_canonical'], keep=False)]
print(duplicates_canonical[['item_name', 'item_canonical']])

In [ ]:
#what are the columns of df_usda_dedup
print(df_usda_dedup.columns)

In [ ]:
#how many single entries are in "item_name" column in df_usda_dedup
single_entries = df_usda_dedup['item_name'].value_counts()

In [ ]:
#are the columns "item_name", "item_norm", and "item_canonical" in df_usda_dedup identical
print(df_usda_dedup['item_name'].equals(df_usda_dedup['item_norm']))
print(df_usda_dedup['item_name'].equals(df_usda_dedup['item_canonical']))
print(df_usda_dedup['item_norm'].equals(df_usda_dedup['item_canonical']))

In [ ]:
#what are the differences in the columns "item_name", "item_norm", and "item_canonical" in df_usda_dedup
differences = df_usda_dedup[df_usda_dedup['item_name'] != df_usda_dedup['item_canonical']][['item_name', 'item_norm', 'item_canonical']]
print(differences.head(20))

In [ ]:
df_usda_dedup

In [ ]:
def remove_duplicate_words(text):
    words = str(text).split()
    seen = set()
    result = []

    for w in words:
        if w not in seen:
            seen.add(w)
            result.append(w)

    return " ".join(result)

df_usda_dedup["item_canonical"] = df_usda_dedup["item_canonical"].apply(remove_duplicate_words)

In [ ]:
df_usda_dedup[df_usda_dedup["item_name"].apply(lambda x: len(str(x).split()) != len(set(str(x).split())))]["item_name"].head(20)

In [ ]:
mask = df_usda_dedup["item_name"].apply(lambda x: len(str(x).split()) != len(set(str(x).split())))
df_usda_dedup.loc[mask, "item_name"] = df_usda_dedup.loc[mask, "item_name"].apply(remove_duplicate_words)

In [ ]:
df_usda_dedup

In [ ]:
#now replace the values of "item_name" column in df_usda_dedup with the values in "item_canonical" column
df_usda_dedup["item_name"] = df_usda_dedup["item_canonical"]

In [ ]:
df_usda_dedup

In [ ]:
#noe remove the "item_canonical" column from df_usda_dedup
df_usda_dedup = df_usda_dedup.drop(columns=["item_canonical"])

In [ ]:
#get also rid of item_norm column in df_usda_dedup
df_usda_dedup = df_usda_dedup.drop(columns=["item_norm"])   

In [ ]:
#show item_name word count in df_usda_dedup
df_usda_dedup['item_name_word_count'] = df_usda_dedup['item_name'].str.split().apply(len)
print(df_usda_dedup['item_name_word_count'].value_counts())

In [ ]:
#show category count in df_usda_dedup
print(df_usda_dedup['cat_word_count'].value_counts())

In [ ]:
#how many single entries are in "item_name" column in df_usda_dedup
single_entries = df_usda_dedup['item_name'].value_counts()
num_single_entries = (single_entries == 1).sum()
print("Number of single entries in 'item_name':", num_single_entries)

In [ ]:
#safe usda dataset with deduplicated item names to csv
df_usda_dedup.to_csv('usda_data_dedup_final.csv', index=False)